![alt text](./Cerny_logo_1.jpg)

# Analysis of Cerny ventilation recordings: `AL000001 - AL000150`

The data processed and analysed in this Notebook were collected by the **Neonatal Emergency and Transport Service of the Peter Cerny Foundation**, Budapest, Hungary

**Author: Dr Gusztav Belteki**

____

This notebook imports all ventilator data of these recordings (including ventilator parameters, settings, alarms (0.5Hz sampling rate) and waveform data (150Hz sampling rate).

- Total: **150 cases**
- 19 cases were removed as they were less than 15 minutes long (empty or partial recordings) 
- **131 cases** remaining

A dictionary containing the processed ventilation data exported as pickle archive: **data_pars_1_150.pickle** 

### 1. Import the required libraries and set options

In [4]:
import IPython
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib
import matplotlib.pyplot as plt

import os
import sys
import pickle
import datetime

from pandas import Series, DataFrame
from datetime import datetime, timedelta

%matplotlib inline
matplotlib.style.use('classic')
matplotlib.rcParams['figure.facecolor'] = 'w'

pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 300)
# This is to turn off a warning message which is given when read_Excel() imports '.xlsx' files
import warnings
warnings.simplefilter("ignore")

In [5]:
print("Python version: {}".format(sys.version))
print("pandas version: {}".format(pd.__version__))
print("matplotlib version: {}".format(matplotlib.__version__))
print("NumPy version: {}".format(np.__version__))
print("IPython version: {}".format(IPython.__version__))

Python version: 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:38:53) [Clang 18.1.8 ]
pandas version: 2.3.2
matplotlib version: 3.10.6
NumPy version: 1.26.4
IPython version: 9.5.0


### 2. List and set the working directory and the directory to write out data

In [7]:
# Name of the external hard drive
DRIVE = 'GB_1'

# Batch containing subsets of recording
BATCH = '1_150'

# Directory on external drive to read the clinical from
DIR_READ = os.path.join(os.sep, 'Volumes', DRIVE, 'Ventilator_data', 'Fabian', f'fabian_ventilator_data_{BATCH}')

# Path to project folder containing ventilation research results
PATH = os.path.join(os.sep, 'Users', 'guszti', 'Library', 'Mobile Documents', 'com~apple~CloudDocs', 
                            'Documents', 'Research', 'Ventilation')

# Folder to export the result of analysis
DIR_WRITE = os.path.join(PATH, 'ventilation_fabian', 'Analyses')
os.makedirs(DIR_WRITE, exist_ok = True)
# Folder containing log files containing errors and anomalies arising during data processing
ERROR_LOGS = os.path.join(DIR_WRITE, 'error_logs')
os.makedirs(ERROR_LOGS, exist_ok = True)

# Folder on a USB stick to export data to and to import processed data exported by other Notebooks
DATA_DUMP = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian',)
os.makedirs(DATA_DUMP, exist_ok = True)

DIR_READ, DIR_WRITE, DATA_DUMP

('/Volumes/GB_1/Ventilator_data/Fabian/fabian_ventilator_data_1_150',
 '/Users/guszti/Library/Mobile Documents/com~apple~CloudDocs/Documents/Research/Ventilation/ventilation_fabian/Analyses',
 '/Volumes/GB_1/data_dump/fabian')

### 3. Import ventilation data as text and create a dictionary of the different recordings

In [9]:
cases = os.listdir(DIR_READ)
cases = sorted(case for case in cases if case.startswith('AL')) # remove hidden other files
len(cases)

150

In [10]:
%%time

# import all data file in the vent_dict dictionary
vent_dict = {}
for case in cases:
    flist = os.listdir(os.path.join(DIR_READ, case))
    for fle in flist:
        # Exclude hidden files and files which are not text files
        if not fle.startswith('.') and fle.endswith('txt'):
            fle_handle = open(os.path.join(DIR_READ, case, fle), 'r', encoding = 'latin1')
            vent_dict[f'{case}_{fle[-5]}'] = fle_handle.read()
            fle_handle.close()

len(vent_dict)

CPU times: user 1 s, sys: 802 ms, total: 1.81 s
Wall time: 4.8 s


309

### 4. Split recordings into records

In [12]:
%%time

with open(os.path.join(ERROR_LOGS, f'{BATCH}_error_log_1.txt'), 'w') as fileout:

    data_dict = {} # In this dict the keys are time points and the values are vent data
    for key, value in sorted(vent_dict.items()):
        data_list = value.split('\n') # Individual records are separated by a 'newline' character
        data_dict[key] = {} # an inner dictionary for the given recording
        for number, record in enumerate(data_list[:-1]):
            try:
                time_stamp, data_str = record.split(';') # splitting record to time stamp and ventilation data
                data_dict[key][time_stamp] = data_str       
            except Exception as e:
                print(f'In %s, record #%d cannot be parsed: \n %s' % (key, number, record[:75]), '\n', e, '\n',  file=fileout)


CPU times: user 2.15 s, sys: 458 ms, total: 2.61 s
Wall time: 2.71 s


### 5. Combine data dictionaries from the same cases

In [14]:
%%time

# data_dict_2 contains the combined ventilation data for each case
data_dict_2 = {}
for case in cases:
    dicts_to_combine = []
    for recording in vent_dict.keys():
        if recording.startswith(case):
            dicts_to_combine.append(data_dict[recording])
    data_dict_2[case] = {k: v for dct in dicts_to_combine for k, v in dct.items()} 

CPU times: user 187 ms, sys: 360 ms, total: 547 ms
Wall time: 688 ms


### 6. Separate parameter data and waves data

In [16]:
%%time

# Records containing parameter data start with '<', records containing waves data always have a space (' ')
# after the first byte (two characters in hexadecimal notation)
with open(os.path.join(ERROR_LOGS, f'{BATCH}_error_log_2.txt'), 'w') as fileout:

    data_dict_waves = {}
    data_dict_pars = {}
    for case in cases:
        data_dict_waves[case] = {}
        data_dict_pars[case] = {}
        for key, value in sorted(data_dict_2[case].items()):
            if value.startswith('<'): # These are ventilator parameter slow data  
                data_dict_pars[case][key] = value[13:-12] # removing 13 characters from the 
                # beginnings and 12 characters from the end (they are not parameters)
            elif value[2] == ' ': # Waves data have a space character at the third position
                data_dict_waves[case][key] = value
            else:
                print(f'In %s, record #%d cannot be parsed: \n %s' % (key, number, record[:75]), '\n',  file=fileout)

CPU times: user 613 ms, sys: 386 ms, total: 999 ms
Wall time: 1.13 s


### 7. Create nested dictionary for the various parameters and their values

In [18]:
%%time

with open(os.path.join(ERROR_LOGS, f'{BATCH}_error_log_3.txt'), 'w') as fileout:
    
    data_dict_pars_2 = {}
    for case in cases:
        data_dict_pars_2[case] = {}
        for time, values in data_dict_pars[case].items():
            time = datetime.strptime(time[:-4], '%Y. %m. %d. %H:%M:%S')
            data_dict_pars_2[case][time] = {} # inner dictionary with the time stamps used as keys
            for pair in values.split('|'):
                if ',' in pair: # ventilator software version field contains a comma
                    continue
                
                try:
                    code, value = pair.split('=') # split records into parameter keys and values
                except:
                    print('Record cannot be unpacked:\n %s, %s\n' % (time, pair), file = fileout)
                
                if code.startswith('0'): # The codes <10 start with zeros (e.g. 00, 01, 02,...)
                                         # and the leading zeros need to be removed
                    code = code[1:]  
                
                try:
                    parameter = int(code)
                except ValueError:
                    print('Error during coverting value to int:\n%r\n' % code, file = fileout)
            
                if code == '145': # Device ID variant, hexadecimal number
                    data_dict_pars_2[case][time][parameter] = value      
                elif code in ['125', '126']: 
                    # convert Mode options 1 & 2 to binary number to retrieve bits
                    data_dict_pars_2[case][time][parameter] = bin(int(value))[2:].zfill(14)  
                elif '.' in value or value == '0': 
                    try:
                        data_dict_pars_2[case][time][parameter] = float(value)
                    except ValueError:
                        print('Value cannot be converted to float\n:%r\n' % value, file = fileout)
                else: 
                    try:
                        data_dict_pars_2[case][time][parameter] = int(value)
                    except:
                        print('Value cannot be converted to int\n:%r\n' % value, file = fileout)

CPU times: user 31.7 s, sys: 578 ms, total: 32.2 s
Wall time: 32.6 s


In [19]:
%%time

# Parameter #125 is 'Mode option 1': its different bits are meaning different parameters
# Parameter #126 is 'Mode option 2': its different bits are meaning different parameters
# See Aculink protocol for more details.
with open(os.path.join(ERROR_LOGS, f'{BATCH}_error_log_4.txt'), 'w') as fileout:

    for case in cases:
        for time in sorted(data_dict_pars_2[case].keys()):
            try:
                # Ventilation_stopped; 0 = no, 1 = yes
                data_dict_pars_2[case][time][270] = int(data_dict_pars_2[case][time][125][-1])
                # VG_state: 0 = off, 1 = on
                data_dict_pars_2[case][time][271] = int(data_dict_pars_2[case][time][125][-2])
                # Volume limit state: 0 = off, 1 = on
                data_dict_pars_2[case][time][272] = int(data_dict_pars_2[case][time][125][-3])
                # Ventilator_range: 0  = neonatal, 1 = paediatric
                data_dict_pars_2[case][time][273] = int(data_dict_pars_2[case][time][125][-4])
                # trigger_mode: 0 = volumetrigger, 1 = flowtrigger
                data_dict_pars_2[case][time][274] = int(data_dict_pars_2[case][time][125][-8])
    
                # I_E_HFOV (HFOV I:E ratio): 0=1:3, 1=1:2, 2=1:1
                if data_dict_pars_2[case][time][125][-14:-12] == '00':
                    data_dict_pars_2[case][time][275] = 0
                elif data_dict_pars_2[case][time][125][-14:-12] == '01':
                    data_dict_pars_2[case][time][275] = 1
                elif data_dict_pars_2[case][time][125][-14:-12] == '10':
                    data_dict_pars_2[case][time][275] = 2
    
                # pressure_rise_control: 0=I-flow, 1=Ramp, 2=AutoIFlow
                if data_dict_pars_2[case][time][126][-2:]   == '00':
                    data_dict_pars_2[case][time][276] = 0
                elif data_dict_pars_2[case][time][126][-2:] == '01':
                    data_dict_pars_2[case][time][276] = 1
                elif data_dict_pars_2[case][time][126][-2:] == '10':
                    data_dict_pars_2[case][time][276] = 2
    
                # HFOV recruitment: 0 = off, 1 = on
                data_dict_pars_2[case][time][277] = int(data_dict_pars_2[case][time][126][-3])
        
            except Exception as e:
                print(f'Error in {case}, {time}', '\n', e, file=fileout)

CPU times: user 739 ms, sys: 568 ms, total: 1.31 s
Wall time: 1.71 s


### 8. Create DataFrame from Parameters Data

In [21]:
%%time

data_pars = {}
for case in cases:
    data_pars[case] = DataFrame(data_dict_pars_2[case]).T

CPU times: user 5.78 s, sys: 964 ms, total: 6.74 s
Wall time: 7.25 s


In [22]:
%%time

# Replace codes for text (see Aculink protocol)
for case in cases:
    a = data_pars[case].copy()
    a= a.replace(-32764, 'off')
    a= a.replace(-32765, 'not valid')
    a= a.replace(-32766, 'out of range')
    a= a.replace(-32767, 'unused')
    data_pars[case] = a

CPU times: user 8.06 s, sys: 715 ms, total: 8.77 s
Wall time: 9.12 s


In [23]:
recording_duration = []
for case in cases:
    # The sampling rate is 0.5 Hz (1 in 2 seconds), e.g. if the lentgh of the DataFrame is 150, its duration is 5 minutes (300 seconds)
    recording_duration.append((case, 2 * len(data_pars[case])))  

recording_duration = DataFrame(recording_duration)
recording_duration.columns = ['case', 'seconds']
recording_duration.index = recording_duration['case']
recording_duration.drop('case', axis = 1, inplace = True)

In [24]:
recording_duration['seconds'].describe()

count      150.000000
mean      4005.773333
std       2708.256692
min          0.000000
25%       2415.500000
50%       3436.000000
75%       5769.500000
max      15676.000000
Name: seconds, dtype: float64

In [25]:
recording_duration.sort_values(by = 'seconds', ascending = True)[:25]

,seconds
case,
AL000001,0
AL000002,0
AL000004,0
AL000046,8
AL000150,10
AL000082,12
AL000027,22
AL000078,32
AL000094,56


### 9. Remove those recordings which are less than 5 minutes long

Recordings less than 5 minutes (300 seconds) long are very likely incomplete and sometimes completely empty.

In [27]:
len(data_pars)

150

In [28]:
# The sampling rate is 0.5 Hz (1 in 2 seconds), if the lentgh of the DataFrame is 150, its duration is 15 minutes (300 seconds)
for case in cases:
    if len(data_pars[case]) < 150:
        print('Removing %s' % case)
        del data_pars[case]
cases = sorted(data_pars.keys())

Removing AL000001
Removing AL000002
Removing AL000004
Removing AL000013
Removing AL000027
Removing AL000039
Removing AL000040
Removing AL000045
Removing AL000046
Removing AL000062
Removing AL000063
Removing AL000078
Removing AL000082
Removing AL000087
Removing AL000088
Removing AL000094
Removing AL000122
Removing AL000124
Removing AL000150


In [29]:
len(data_pars)

131

In [30]:
len(cases)

131

### 10. Replace codes for categorical variables with informative category names

In [32]:
mapping_vent_mode = {0: None, 1: 'IPPV', 2: 'SIPPV', 3: 'SIMV', 4: 'SIMVPSV', 5: 'PSV', 
                     6: 'CPAP', 7: 'NCPAP', 8: 'DUOPAP', 9: 'HFO', 10: 'O2therapy', 15: 'service'}
for case in cases:
    data_pars[case][101].replace(mapping_vent_mode, inplace = True)

In [33]:
mapping_patient_range = {1: 'Neonatal', 2: 'Pediatric'}
for case in cases:
    data_pars[case][100].replace(mapping_patient_range, inplace = True)

In [34]:
mapping_patient_range_2 = {0: 'Neonatal', 1: 'Pediatric'}
for case in cases:
    data_pars[case][273].replace(mapping_patient_range_2, inplace = True)

In [35]:
mapping_power = {0: 'Network', 1: 'Battery'}
for case in cases:
    data_pars[case][127].replace(mapping_power, inplace = True)

In [36]:
mapping_off_on = {0: 'off', 1: 'on'}
for case in cases:
    data_pars[case][157].replace(mapping_off_on, inplace = True)
    data_pars[case][158].replace(mapping_off_on, inplace = True)
    data_pars[case][271].replace(mapping_off_on, inplace = True)
    data_pars[case][272].replace(mapping_off_on, inplace = True)
    data_pars[case][277].replace(mapping_off_on, inplace = True)

In [37]:
mapping_no_yes = {0: 'no', 1: 'yes'}
for case in cases:
    data_pars[case][270].replace(mapping_no_yes, inplace = True)

In [38]:
mapping_trigger = {0: 'Volumetrigger', 1: 'Flowtrigger'}
for case in cases:
    data_pars[case][274].replace(mapping_trigger, inplace = True)

In [39]:
mapping_IE_HFOV = {0: '1:3', 1: '1:2', 2: '1:1'}
for case in cases:
    data_pars[case][275].replace(mapping_IE_HFOV, inplace = True)

In [40]:
mapping_pressure_rise_ctrl = {0: 'I-flow', 1: 'Ramp', 2: 'AutoIFlow'}
for case in cases:
    data_pars[case][276].replace(mapping_pressure_rise_ctrl, inplace = True)

In [41]:
mapping_pressure_unit = {0: 'mbar', 1: 'cmH2O',}
for case in cases:
    data_pars[case][140].replace(mapping_pressure_unit, inplace = True)

In [42]:
mapping_CO2 = {0: 'mmHg', 1: 'kPa', 2: 'Vol%'}
for case in cases:
    data_pars[case][141].replace(mapping_CO2, inplace = True)

### 11. Parse the parameter values using Fabian parameter library

In [44]:
par_key_table = pd.read_excel(os.path.join(PATH, 'ventilation_fabian', 'Fabian_parameters.xlsx'))

In [45]:
par_key_table;

In [46]:
par_key_dict = {}
for row in par_key_table.index:
    par_key_dict[par_key_table.code[row]] = par_key_table.name[row]

In [47]:
for case in cases:
    data_pars[case].rename(columns = par_key_dict, inplace = True )

In [48]:
# Sort DataFrames according to time stamp index
for case in cases:
    data_pars[case].sort_index(inplace = True)

### 12. Write individual text files with the ventilator modes

##### Create sub-directories for each case if it does not yet exist

In [51]:
# Images and raw data will be written on an external hard drive
if not os.path.isdir(os.path.join(DATA_DUMP, 'fabian_cases')):
    os.makedirs(os.path.join(DATA_DUMP, 'fabian_cases'))

for case in cases: 
    if not os.path.isdir(os.path.join(DATA_DUMP, 'fabian_cases', case)):
        os.makedirs(os.path.join(DATA_DUMP, 'fabian_cases', case))

In [52]:
for case in cases:
    a = data_pars[case]
    o2therapy = len(a[a['Ventilator_mode'] == 'O2therapy'])
    ncpap = len(a[a['Ventilator_mode'] == 'NCPAP'])
    duopap = len(a[a['Ventilator_mode'] == 'DUOPAP'])
    simv = len(a[a['Ventilator_mode'] == 'SIMV'])
    ippv = len(a[a['Ventilator_mode'] == 'IPPV'])
    sippv = len(a[a['Ventilator_mode'] == 'SIPPV'])
    simvpsv = len(a[a['Ventilator_mode'] == 'SIMVPSV'])
    vg_on = len(data_pars[case][data_pars[case]['VG_state'] == 'on'])
  
    fileout = open(os.path.join(DATA_DUMP, 'fabian_cases', case, f'{case}_vent_info.txt'), 'w') 
    fileout.write('O2 therapy: %d sec \n' % (o2therapy * 2))
    fileout.write('NCPAP:      %d sec \n' % (ncpap * 2))
    fileout.write('DUOPAP:     %d sec \n' % (duopap * 2))
    fileout.write('IPPV:       %d sec \n' % (ippv * 2))
    fileout.write('SIPPV:      %d sec \n' % (sippv * 2))
    fileout.write('SIMV:       %d sec \n' % (simv * 2))
    fileout.write('SIMVPSV:    %d sec \n\n' % (simvpsv * 2))
    fileout.write('VG on:      %d sec \n' % (vg_on * 2))
    fileout.close()

### Export processed data as pickle files

In [54]:
with open(os.path.join(DATA_DUMP, f'data_pars_{BATCH}.pickle'), 'wb') as handle:
    pickle.dump(data_pars, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [55]:
with open(os.path.join(DATA_DUMP, f'cases_{BATCH}.pickle'), 'wb') as handle:
    pickle.dump(cases, handle, protocol=pickle.HIGHEST_PROTOCOL)